<a href="https://colab.research.google.com/github/bgq1011/DeepLearning_MaskDetection_Nhom8_SwinTransformer/blob/main/DeepLearning_MaskDetection_Nhom8_SwinTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import kagglehub
shiekhburhan_face_mask_dataset_path = kagglehub.dataset_download('shiekhburhan/face-mask-dataset')
alincijov_swin_tiny_small_22k_pretrained_path = kagglehub.dataset_download('alincijov/swin-tiny-small-22k-pretrained')

print('Data source import complete.')


In [ ]:
import os

# Đường dẫn folder ảnh
DATASET_PATH = "/kaggle/input/datasets/shiekhburhan/face-mask-dataset/FMD_DATASET"

# Đường dẫn file trọng số Swin Tiny
WEIGHTS_PATH = "/kaggle/input/datasets/alincijov/swin-tiny-small-22k-pretrained/swin_tiny_patch4_window7_224_22k.pth"

if os.path.exists(DATASET_PATH):
    print("Đã kết nối folder:", os.listdir(DATASET_PATH))

In [ ]:
!pip install timm ultralytics -q

In [ ]:
import torch
import torch.nn as nn
import timm
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_model():
    # Khởi tạo Swin Tiny
    model = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=3)

    # Đóng băng các lớp trước
    for param in model.parameters():
        param.requires_grad = False
    # 3. Mở khóa lớp Head mặc định của timm để học
    for param in model.head.parameters():
        param.requires_grad = True
    return model.to(device)

model = get_model()
print("Model đã sẵn sàng và đã đóng băng backbone!")

In [ ]:
import torch
import torch.nn as nn
import timm
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Định nghĩa các bước tiền xử lý
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),       # Swin Transformer yêu cầu ảnh 224x224
        transforms.RandomHorizontalFlip(p=0.5), # Lật ảnh ngẫu nhiên (người đứng bên trái/phải)
        transforms.RandomRotation(15),      # Xoay ảnh nhẹ (người nghiêng đầu)
        transforms.ColorJitter(brightness=0.2, contrast=0.2), # Thay đổi độ sáng/tối (quan trọng cho thực tế)
        transforms.ToTensor(),              # Chuyển ảnh về dạng Tensor để máy học
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Chuẩn hóa theo ImageNet
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Load dữ liệu từ folder

full_dataset = datasets.ImageFolder(DATASET_PATH) # Load gốc chưa transform

# Chia 80% train, 20% val
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_raw, val_raw = torch.utils.data.random_split(full_dataset, [train_size, val_size])

# Áp dụng transform riêng cho từng bộ
class ApplyTransform(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform: x = self.transform(x)
        return x, y
    def __len__(self):
        return len(self.subset)

train_data = ApplyTransform(train_raw, transform=data_transforms['train'])
val_data = ApplyTransform(val_raw, transform=data_transforms['val'])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)

print(f"Đã tiền xử lý: Resizing, Flip, Rotation, ColorJitter cho {len(train_data)} ảnh train.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def imshow(img):
    img = img.clone().detach().cpu()
    inv_normalize = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225]
    )
    img = inv_normalize(img)
    img = np.clip(img.numpy(), 0, 1)
    plt.imshow(np.transpose(img, (1, 2, 0)))
    plt.axis('off')

# Lấy dữ liệu
images, labels = next(iter(train_loader))
class_names = full_dataset.classes

plt.figure(figsize=(20, 12))
for i in range(15):
    plt.subplot(3, 5, i + 1)
    imshow(images[i])
    plt.title(f"Label: {class_names[labels[i]]}", color='darkred', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import timm
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Hàm huấn luyện và vẽ biểu đồ lịch sử quá trình học
def train_and_plot(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    since = time.time()

    # Chứa dữ liệu để vẽ biểu đồ
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

# Vòng lặp chính chạy qua từng Epoch
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch + 1}/{num_epochs}')
        print('-' * 10)

# Trong mỗi Epoch, thực hiện luân phiên 2 pha: Huấn luyện (Train) và Kiểm thử (Val)
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Thiết lập mô hình ở chế độ huấn luyện
                current_loader = train_loader
            else:
                model.eval()   # Thiết lập mô hình ở chế độ đánh giá
                current_loader = val_loader

            running_loss = 0.0 # Tổng sai số trong Epoch hiện tại
            running_corrects = 0 # Tổng số mẫu dự đoán đúng trong Epoch hiện tại

            pbar = tqdm(current_loader, desc=f"{phase} Epoch {epoch+1}")

            # Vòng lặp qua các batch dữ liệu
            for inputs, labels in pbar:
                inputs = inputs.to(device)
                labels = labels.to(device).long()

                optimizer.zero_grad()

                # Theo dõi luồng tính toán
                with torch.set_grad_enabled(phase == 'train'):
                    # Đưa ảnh qua mô hình (Forward pass) để lấy kết quả dự đoán
                    outputs = model(inputs)

                    if outputs.dim() > 2:
                        outputs = outputs.view(outputs.size(0), -1)

                    _, preds = torch.max(outputs, 1) # Lấy chỉ số của lớp có xác suất cao nhất
                    loss = criterion(outputs, labels)  # Tính toán sai số giữa dự đoán và thực tế

                    # Nếu đang ở phase huấn luyện, thực hiện lan truyền ngược và tối ưu
                    if phase == 'train':
                        loss.backward() #Tính toán đạo hàm (Backward pass)
                        optimizer.step() # Cập nhật trọng số dựa trên đạo hàm vừa tính

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)


            # Tính toán sai số trung bình và độ chính xác trung bình cho toàn bộ Epoch
            epoch_loss = running_loss / len(current_loader.dataset)
            epoch_acc = running_corrects.double() / len(current_loader.dataset)

            # Lưu vào history để vẽ biểu đồ
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())

                # Lưu lại trọng số tốt nhất nếu độ chính xác tăng
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

    # Tính tổng thời gian hoàn thành quá trình huấn luyện
    time_elapsed = time.time() - since
    print(f'\nHuấn luyện hoàn tất trong {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Độ chính xác cao nhất (Best Val Acc): {best_acc:4f}')

    # Nạp lại trọng số tốt nhất cho mô hình
    model.load_state_dict(best_model_wts)
    return model, history

# --- THIẾT LẬP THAM SỐ  ---
# 1. Cấu hình các tham số huấn luyện
#Định nghĩa hàm mất mát
criterion = nn.CrossEntropyLoss()
# Dùng AdamW với lr=1e-4:Learning rate thấp giúp tinh chỉnh
#lớp Head mà không làm hỏng kiến thức cũ của mô hình(vì đang fine-tuning)
optimizer = torch.optim.AdamW(model.head.parameters(), lr=1e-4)

# 2. chạy với vòng lặp = 10
model_ft, history = train_and_plot(model, train_loader, val_loader, criterion, optimizer, num_epochs=10)

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Biểu đồ Loss
    ax1.plot(history['train_loss'], label='Train Loss', marker='o')
    ax1.plot(history['val_loss'], label='Val Loss', marker='o')
    ax1.set_title('Đồ thị Loss qua các Epoch')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)

    # Biểu đồ Accuracy
    ax2.plot(history['train_acc'], label='Train Acc', marker='o')
    ax2.plot(history['val_acc'], label='Val Acc', marker='o')
    ax2.set_title('Đồ thị Accuracy qua các Epoch')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True)

    plt.show()

plot_history(history)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import numpy as np

def evaluate_model(model, val_loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            if outputs.dim() > 2:
                outputs = outputs.view(outputs.size(0), -1)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Lấy tên các lớp từ dataset
    class_names = val_loader.dataset.subset.dataset.classes

    # 1. In báo cáo chi tiết
    print("BÁO CÁO PHÂN LOẠI ")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # 2. Vẽ Ma trận nhầm lẫn
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.ylabel('Thực tế (True Label)')
    plt.xlabel('Dự đoán (Predicted Label)')
    plt.show()

evaluate_model(model_ft, val_loader)

In [ ]:
# Lưu toàn bộ trọng số tốt nhất vào file .pth
torch.save(model_ft.state_dict(), 'swin_mask_detector_bgq.pth')
print("Đã lưu model thành công")